# Analysis 4 — Prediction-Timing Reframing (Final Synthesis)
**담당**: Donghyun (Part B 확장 — 콘텐츠/채널 트랙 → 예측 시점 재프레이밍)

## 목적
- **가설 H_A4**: 예측 정확도는 콘텐츠/채널 피처의 종류가 아니라 "예측 시점"의 단조 함수다.
- 예측 시점을 업로드(T0/T1) → 트렌딩 직후(T2) → 참여 발생 후(T3)로 옮길수록 R²가 계단식으로 상승함을 실측 증명한다.
- T1 (Content + Channel)의 unexplained variance 70%를 사후 정보 회수분과 구조적 데이터 한계분으로 정량 분해한다.

> **구조**: 본 분석의 1급 단위는 **시점 사다리(Timing Ladder) T0→T3**다. Prep/Decomposition/Drift/Visualization/Validation은 사다리를 떠받치는 부속 단계다.

## Prep — 3원 머지 & 검증
- base 피처 (`features_partB_v2.csv`) + channel 피처 (`step1_channel_features.csv`) + post-upload 사후 피처 (`cleaned_USvideos.csv`)를 머지합니다.
- `hour_bin`을 ordered Categorical로 설정하고, `comments_disabled`/`ratings_disabled`를 int로 변환합니다.

In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import matplotlib.pyplot as plt
import seaborn as sns

RS = 42
os.makedirs("out", exist_ok=True)

print("Prep: Loading and merging datasets...")
base = pd.read_csv("../Analysis1_title/LR/out/features_partB_v2.csv", parse_dates=["publish_time"])
chan = pd.read_csv("../Analysis3_channel/out/step1_channel_features.csv")[["video_id", "chan_mean_oof", "chan_mean_naive", "chan_freq"]]

# Merge base and channel
df = base.merge(chan, on="video_id", how="inner")
assert len(df) == len(base), f"Row count mismatch: {len(df)} vs {len(base)}"
df["chan_freq_log"] = np.log1p(df["chan_freq"])

# Load post-upload features
post = pd.read_csv("../dataset/cleaned_USvideos.csv")[["video_id", "trending_lag", "days_on_trending", "log_likes", "log_comment_count", "comments_disabled", "ratings_disabled"]]

# Merge base+channel and post-upload
df = df.merge(post, on="video_id", how="inner")
assert len(df) == len(base)

# Convert boolean columns to integer
df["comments_disabled"] = df["comments_disabled"].astype(int)
df["ratings_disabled"] = df["ratings_disabled"].astype(int)

# Set hour_bin as an ordered Categorical
hour_labels = ["00-05", "06-11", "12-17", "18-23"]
df["hour_bin"] = pd.Categorical(df["hour_bin"], categories=hour_labels, ordered=True)

df.to_csv("out/timing_merged.csv", index=False)
print(f"Merged dataset shape: {df.shape}. Saved to out/timing_merged.csv")

Prep: Loading and merging datasets...
Merged dataset shape: (6249, 28). Saved to out/timing_merged.csv


## Timing Ladder — 4단 시점 사다리 T0→T3 (핵심)
- OLS (LR), RF, GBM 모델을 T0~T3 각 4단(rung)에 대해 학습시키고 R² 점수를 계산합니다.
- 80/20 random split (`random_state=42`) 및 category 필터를 일관되게 적용합니다.
- T0=콘텐츠, T1=+채널(업로드시점 천장), T2=+트렌딩, T3=+참여도.

In [2]:
# Helper Functions
def evaluate_model(model, features, df, split_type="random", is_formula=True):
    df_valid = df.dropna(subset=["publish_time"]).reset_index(drop=True)
    
    if split_type == "random":
        tr, te = train_test_split(df_valid, test_size=0.2, random_state=RS)
    elif split_type == "time":
        # Chronological split: sort by publish_time before cutting (matches A3)
        df_sorted = df_valid.sort_values("publish_time").reset_index(drop=True)
        cut = int(len(df_sorted) * 0.8)
        tr = df_sorted.iloc[:cut].copy()
        te = df_sorted.iloc[cut:].copy()
        
    tr_cats = set(tr["category"].unique())
    te = te[te["category"].isin(tr_cats)].copy()
    
    if is_formula:
        m = smf.ols(features, data=tr).fit()
        pred_te = m.predict(te)
        return r2_score(te["log_views"], pred_te)
    else: 
        X_tr = tr[features]
        y_tr = tr["log_views"]
        X_te = te[features]
        y_te = te["log_views"]
        
        model.fit(X_tr, y_tr)
        return r2_score(y_te, model.predict(X_te))

formulas = {
    "T0": "log_views ~ title_len + caps_ratio + exclaim_cnt + question_cnt + has_number + has_bracket + tag_cnt + C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday) + C(category)",
    "T1": "log_views ~ title_len + caps_ratio + exclaim_cnt + question_cnt + has_number + has_bracket + tag_cnt + chan_mean_oof + chan_freq_log + C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday) + C(category)",
    "T2": "log_views ~ title_len + caps_ratio + exclaim_cnt + question_cnt + has_number + has_bracket + tag_cnt + chan_mean_oof + chan_freq_log + trending_lag + days_on_trending + C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday) + C(category)",
    "T3": "log_views ~ title_len + caps_ratio + exclaim_cnt + question_cnt + has_number + has_bracket + tag_cnt + chan_mean_oof + chan_freq_log + trending_lag + days_on_trending + log_likes + log_comment_count + comments_disabled + ratings_disabled + C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday) + C(category)"
}

obj = ["title_len", "caps_ratio", "exclaim_cnt", "question_cnt", "has_number", "has_bracket", "tag_cnt"]
cat_dummies = pd.get_dummies(df[["category", "hour_bin", "publish_weekday"]].astype(str), drop_first=True).astype(float)

for col in cat_dummies.columns:
    df[col] = cat_dummies[col]

F_matrix_T0 = obj + list(cat_dummies.columns)
F_matrix_T1 = F_matrix_T0 + ["chan_mean_oof", "chan_freq_log"]
F_matrix_T2 = F_matrix_T1 + ["trending_lag", "days_on_trending"]
F_matrix_T3 = F_matrix_T2 + ["log_likes", "log_comment_count", "comments_disabled", "ratings_disabled"]

matrix_features = {"T0": F_matrix_T0, "T1": F_matrix_T1, "T2": F_matrix_T2, "T3": F_matrix_T3}

rungs = ["T0", "T1", "T2", "T3"]
descs = {"T0": "Content Features", "T1": "Content + Channel", "T2": "Content + Channel + Trending Meta", "T3": "Content + Channel + Trending Meta + Engagement"}
observables = {"T0": "Upload Time (Before Upload)", "T1": "Upload Time (Before Upload)", "T2": "Trending Time (Post-Upload)", "T3": "Trending Time (Post-Upload)"}

rf_model = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RS, n_jobs=-1)
gbm_model = GradientBoostingRegressor(random_state=RS)

results = []
for rung in rungs:
    lr_r2 = evaluate_model(None, formulas[rung], df, split_type="random", is_formula=True)
    rf_r2 = evaluate_model(rf_model, matrix_features[rung], df, split_type="random", is_formula=False)
    gbm_r2 = evaluate_model(gbm_model, matrix_features[rung], df, split_type="random", is_formula=False)
    lr_time_r2 = evaluate_model(None, formulas[rung], df, split_type="time", is_formula=True)
    
    results.append({
        "rung": rung,
        "features_desc": descs[rung],
        "observable_at": observables[rung],
        "LR_R2": lr_r2,
        "RF_R2": rf_r2,
        "GBM_R2": gbm_r2,
        "LR_time_R2": lr_time_r2
    })

ladder_df = pd.DataFrame(results)
ladder_df.to_csv("out/timing_ladder_r2.csv", index=False)
print("\nTiming Ladder Results:")
print(ladder_df.to_string(index=False))


Timing Ladder Results:
rung                                  features_desc               observable_at    LR_R2    RF_R2   GBM_R2  LR_time_R2
  T0                               Content Features Upload Time (Before Upload) 0.118790 0.111509 0.127373   -1.271101
  T1                              Content + Channel Upload Time (Before Upload) 0.295598 0.303557 0.324061   -0.568000
  T2              Content + Channel + Trending Meta Trending Time (Post-Upload) 0.428138 0.517002 0.512113   -0.143762
  T3 Content + Channel + Trending Meta + Engagement Trending Time (Post-Upload) 0.840816 0.857677 0.857859    0.716991


## Decomposition — 잔여분산 분해 (70% 해부)
- T1 OLS R² 기준, unexplained variance 70%가 사후정보 회수분과 구조적 잔존분으로 어떻게 분해되는지 계산합니다.

In [3]:
r2_t1 = ladder_df.loc[ladder_df["rung"] == "T1", "LR_R2"].values[0]
r2_t3 = ladder_df.loc[ladder_df["rung"] == "T3", "LR_R2"].values[0]

total_unexplained_t1 = 1.0 - r2_t1
recovered_var = r2_t3 - r2_t1
structural_residual = 1.0 - r2_t3

decomp_data = [
    {"component": "Explained by T1 (Content + Channel)", "variance_pct": r2_t1},
    {"component": "Recovered by Post-Upload (T3 - T1)", "variance_pct": recovered_var},
    {"component": "Structural Residual (Unexplained)", "variance_pct": structural_residual}
]
decomp_df = pd.DataFrame(decomp_data)
decomp_df.to_csv("out/timing_variance_decomp.csv", index=False)
print("\nVariance Decomposition of Target (log_views):")
print(decomp_df.to_string(index=False))


Variance Decomposition of Target (log_views):
                          component  variance_pct
Explained by T1 (Content + Channel)      0.295598
 Recovered by Post-Upload (T3 - T1)      0.545218
  Structural Residual (Unexplained)      0.159184


## Drift — 시간 일반화 진단 & RF R²=0.245 재라벨링
- 랜덤 스플릿(Random Split) vs 시간 기반 스플릿(Time Split) 성능을 비교하여 temporal drift가 시점 사다리를 따라 어떻게 회복되는지 확인합니다.

In [4]:
print("Drift: Comparing Random Split vs Time Split")
for idx, r in ladder_df.iterrows():
    print(f"Rung {r['rung']} ({r['features_desc']}):")
    print(f"  Random split test R²: {r['LR_R2']:.4f}")
    print(f"  Time split test R²:   {r['LR_time_R2']:.4f}")

Drift: Comparing Random Split vs Time Split
Rung T0 (Content Features):
  Random split test R²: 0.1188
  Time split test R²:   -1.2711
Rung T1 (Content + Channel):
  Random split test R²: 0.2956
  Time split test R²:   -0.5680
Rung T2 (Content + Channel + Trending Meta):
  Random split test R²: 0.4281
  Time split test R²:   -0.1438
Rung T3 (Content + Channel + Trending Meta + Engagement):
  Random split test R²: 0.8408
  Time split test R²:   0.7170


## Visualization — 사다리 곡선 & 분해
- T0~T3 단조 상승 사다리 곡선과 잔여 분산 분해 스택 바 차트를 생성합니다.

In [5]:
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── Fig 1: Ladder Curve ──
plt.figure(figsize=(10, 6))
x_labels = ["T0: Content", "T1: Content+Channel", "T2: +Trending Meta", "T3: +Engagement"]

plt.plot(x_labels, ladder_df["LR_R2"], marker='o', color='#3B4FE4', linewidth=2.5, label='Linear Regression (OLS)')
plt.plot(x_labels, ladder_df["RF_R2"], marker='s', color='#1A7F5A', linewidth=2, linestyle='--', label='Random Forest')
plt.plot(x_labels, ladder_df["GBM_R2"], marker='^', color='#E8B84B', linewidth=2, linestyle=':', label='Gradient Boosting')

plt.axvspan(-0.2, 1.2, color='#ECEFF1', alpha=0.5, label='Pre-Upload (Strict Non-Leakage)')
plt.axvspan(1.2, 3.2, color='#FFF9C4', alpha=0.3, label='Post-Upload (Simulated Timing Shift)')

plt.title("Prediction Timing Ladder: R² Monotonic Increase", fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Test R²", fontsize=11)
plt.xlabel("Prediction Timing (Rung)", fontsize=11)
plt.ylim(0, 1.0)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig("out/fig_timing_ladder.png")
plt.close()
print("Saved: out/fig_timing_ladder.png")

# ── Fig 2: Variance Decomposition Stacked Bar ──
plt.figure(figsize=(6, 6))
labels = ['Total Variance (100%)']
t1_exp = [r2_t1]
post_rec = [recovered_var]
residual = [structural_residual]

plt.bar(labels, t1_exp, label='Explained by T1 (Content + Channel)', color='#3B4FE4', width=0.4)
plt.bar(labels, post_rec, bottom=t1_exp, label='Recovered by Post-Upload info (T3 - T1)', color='#1A7F5A', width=0.4)
plt.bar(labels, residual, bottom=np.array(t1_exp) + np.array(post_rec), label='Structural Residual (Unobserved)', color='#C8CDD6', width=0.4)

plt.text(0, r2_t1 / 2, f"{r2_t1*100:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=10)
plt.text(0, r2_t1 + recovered_var / 2, f"{recovered_var*100:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=10)
plt.text(0, r2_t1 + recovered_var + structural_residual / 2, f"{structural_residual*100:.1f}%", ha='center', va='center', color='black', fontweight='bold', fontsize=10)

plt.title("Variance Decomposition of YouTube Views (log_views)", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Variance Percentage", fontsize=11)
plt.ylim(0, 1.05)
plt.legend(loc='upper right', fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.savefig("out/fig_timing_variance.png")
plt.close()
print("Saved: out/fig_timing_variance.png")

Saved: out/fig_timing_ladder.png
Saved: out/fig_timing_variance.png


## Validation — T0/T1 재현 검증
- OLS T0 R²와 T1 R²가 기존 분석 수치(0.1188 / 0.2956)와 일치하는지 자동으로 검증합니다.

In [6]:
t0_r2_lr = ladder_df.loc[ladder_df["rung"] == "T0", "LR_R2"].values[0]
t1_r2_lr = ladder_df.loc[ladder_df["rung"] == "T1", "LR_R2"].values[0]

t0_target = 0.1188
t1_target = 0.2956

print(f"T0 LR R2: {t0_r2_lr:.4f} (Target: {t0_target:.4f})")
print(f"T1 LR R2: {t1_r2_lr:.4f} (Target: {t1_target:.4f})")

assert abs(t0_r2_lr - t0_target) < 1e-3, f"T0 mismatch"
assert abs(t1_r2_lr - t1_target) < 1e-3, f"T1 mismatch"
print("Success: T0 and T1 OLS R2 match A3 baseline exactly!")

is_monotonic = all(ladder_df["LR_R2"].iloc[i] <= ladder_df["LR_R2"].iloc[i+1] for i in range(len(ladder_df)-1))
if is_monotonic:
    print("Success: Monotonic relationship confirmed (T0 <= T1 <= T2 <= T3)!")
else:
    print("Warning: Monotonic relationship not strictly observed.")

T0 LR R2: 0.1188 (Target: 0.1188)
T1 LR R2: 0.2956 (Target: 0.2956)
Success: T0 and T1 OLS R2 match A3 baseline exactly!
Success: Monotonic relationship confirmed (T0 <= T1 <= T2 <= T3)!
